# 20. Ders: Yüz Tespiti (Face Detection) - Piksellerde İnsan Aramak

## Adım 1: Teorik Çerçeve ve Günlük Hayat Analojisi

Telefonunuzun kamerasını arkadaşınıza tuttuğunuzda anında yüzünün etrafında sarı bir kutucuk (odak kutusu) belirir. Hatta fotoğrafı çekerken arkadaşınız gözünü kırptıysa telefonunuz sizi uyarır. Peki o ufacık telefon lensi, karşınızda bir duvar, ağaç veya kedi değil de bir 'İnsan Yüzü' olduğunu nereden biliyor?

Yüz Bulma (Face Detection) teknolojisi, resimdeki piksellerin ışık ve karanlık değerlerini (örneğin göz çukurları karanlıktır, burun kemiği aydınlıktır) matematiksel şablonlarla üst üste koyarak insan yüzü formunu arayan efsanevi bir algoritmadır. Bu dersimizde bu tarihi algoritmayı (Haar Cascades) kullanarak fotoğraflardaki insanları tespit edeceğiz.

## Adım 2: Teknik Terimler ve Sözlük

- Face Detection (Yüz Tespiti): Sadece 'Burada bir insan yüzü var' diyebilen sistem (Kim olduğunu bilmez).
- Face Recognition (Yüz Tanıma): Bulunan yüzün 'Ali'ye veya 'Ayşe'ye ait olduğunu bilebilen gelişmiş (Face ID gibi) sistemler.
- Bounding Box (Sınır Kutusu): Tespit edilen bir nesnenin etrafına çizilen renkli dikdörtgen veya kare kutu.
- Haar Cascades: 2001 yılında Viola ve Jones tarafından icat edilen, yüz binlerce suratla eğitilmiş ve yüzün ışık/gölge haritasını çıkarmış ünlü yüz tespit algoritması.

## Adım 3: İnsanların Olduğu Örnek Bir Fotoğraf Yükleme

İçinde birkaç kişinin olduğu kalabalık bir fotoğraf indirelim ve OpenCV ile okuyalım.

In [ ]:
import cv2
import urllib.request
import matplotlib.pyplot as plt

# İki kişinin bulunduğu ünlü bir stok fotoğraf
url = "https://upload.wikimedia.org/wikipedia/commons/thumb/e/e5/NASA_Apollo_11_Press_Conference_-_GPN-2000-001659.jpg/800px-NASA_Apollo_11_Press_Conference_-_GPN-2000-001659.jpg"
dosya_adi = "astronotlar.jpg"

urllib.request.urlretrieve(url, dosya_adi)
resim = cv2.imread(dosya_adi)
resim_rgb = cv2.cvtColor(resim, cv2.COLOR_BGR2RGB)

plt.figure(figsize=(8,6))
plt.imshow(resim_rgb)
plt.axis("off")
plt.title("Orijinal Fotoğraf (Apollo 11 Ekibi)")
plt.show()

## Adım 4: Haar Cascade Şablonunu (Yüz Haritasını) Yükleme

OpenCV'nin içinde önceden eğitilmiş bir XML (kurallar) dosyası vardır. Biz kodu yazdığımızda o dosya gidip piksellerdeki karanlık ve aydınlık bölgelere bakarak gözü, burnu ve ağzı arayacaktır.

In [ ]:
# Colab ortamında OpenCV'nin hazır modelleri cv2.data.haarcascades altında bulunur.
# Frontalface (Önden görünüm) şablonunu yüklüyoruz.
yuz_bulucu = cv2.CascadeClassifier(cv2.data.haarcascades + 'haarcascade_frontalface_default.xml')

print("Yüz tespit zekası başarıyla yüklendi.")

## Adım 5: Görüntüyü Siyah-Beyaza Çevirme (Zorunlu)

Haar algoritmaları renklerle (Örn: Ten rengi) ilgilenmez. Işığın düştüğü yerlerle ilgilenir. Bu yüzden performansı artırmak için resmi Gri (Gray) tona çevirmek zorundayız.

In [ ]:
gri_resim = cv2.cvtColor(resim, cv2.COLOR_BGR2GRAY)

print("Algoritmanın daha hızlı çalışması için resim Gri tona (matrise) çevrildi.")

## Adım 6: Algoritmayı Çalıştırıp Yüzlerin Koordinatlarını Bulma

Sihirli fonksiyonumuz `detectMultiScale`. Resimde yüz varsa, bize yüzün başladığı (x,y) koordinatlarını ve genişlik(w), yükseklik(h) değerlerini verecek.

In [ ]:
# scaleFactor: Resmin küçültülerek taranma oranı (Genelde 1.1 veya 1.2 kullanılır)
# minNeighbors: Bir karenin yüz sayılması için etrafında kaç tane pozitif komşusu olması gerektiği (Sahte yüzleri önler)
bulunan_yuzler = yuz_bulucu.detectMultiScale(gri_resim, scaleFactor=1.2, minNeighbors=4)

print(f"Taramalar Sonucu Toplam {len(bulunan_yuzler)} Adet Yüz Bulundu!")
print("Koordinatlar (X, Y, Genişlik, Yükseklik):")
print(bulunan_yuzler)

## Adım 7: Yüzlerin Etrafına Sınır Kutusu (Bounding Box) Çizme

Elimizdeki (x, y, w, h) verileriyle for döngüsü kullanarak orijinal **renkli resim** üzerine kırmızı dikdörtgenler çizeceğiz.

In [ ]:
# Orjinal RGB resmimizin bir kopyasını alalım ki orjinali bozulmasın
cizimli_resim = resim_rgb.copy()

for (x, y, w, h) in bulunan_yuzler:
    # cv2.rectangle(resim, baslangic_koordinati, bitis_koordinati, renk(RGB), cizgi_kalinligi)
    # x+w (başlangıç x'e genişliği ekle), y+h (başlangıç y'ye yüksekliği ekle)
    cv2.rectangle(cizimli_resim, (x, y), (x + w, y + h), (255, 0, 0), 3)
    
print("Kutular çizildi.")

## Adım 8: Sonucu Görüntüleme

Yapay zekanın eserini Matplotlib ile ekranda görelim.

In [ ]:
plt.figure(figsize=(10,8))
plt.imshow(cizimli_resim)
plt.axis("off")
plt.title(f"Yapay Zeka {len(bulunan_yuzler)} Yüz Tespit Etti")
plt.show()

## Adım 9: Dene ve Keşfet - Hatalı Taramalar (False Positives/Negatives)

Adım 6'daki `minNeighbors` değerini `1` yapıp kodu tekrar çalıştırın. Büyük ihtimalle mikrofonu veya arka plandaki gölgeleri de yüz zannedecektir (False Positive - Yanlış Pozitif). 

Peki ya `minNeighbors` değerini `20` yaparsak ne olur? Bu sefer gerçekten var olan yüzleri de kaçıracaktır (False Negative). İşte yapay zeka mühendisliği, bu parametreleri en kusursuz ayara getirme mesleğidir.

## Adım 10: Sıra Sende Görevi - Profil (Yandan Görünüm) Tespiti

Problem: Haar cascade modelleri çok katıdır. 'FrontalFace' (önden) modeli, yandan bakan bir profili asla yüz olarak tanımaz. Yandan bakan insanları bulmak için OpenCV'nin 'ProfileFace' şablonunu kullanmanız isteniyor.

Gereksinimler:
1. Adım 4'te kullandığımız `haarcascade_frontalface_default.xml` dosyası yerine, yandan bakan yüzler için eğitilmiş olan `haarcascade_profileface.xml` dosyasını kullanarak `yandan_yuz_bulucu` adında yeni bir Classifier tanımlayın.
2. Bu yeni bulucuyu kullanarak `gri_resim` üzerinde tarama (`detectMultiScale`) yapın.
3. Bulunan yüzlerin (yandan yüzlerin) etrafına for döngüsü ile YEŞİL `(0, 255, 0)` renkli kutular çizin (Bunun için `cizimli_resim`'in kopyasını kullanabilirsiniz).
4. Sonucu ekranda gösterin. Not: Apollo 11 fotoğrafındaki kişiler genellikle öne dönük veya 3/4 açılı olduğundan, 'profileface' hepsini bulamayabilir. Bu gayet normal bir AI kısıtlamasıdır.

In [ ]:
# GÖREV KODLARINIZI BURAYA YAZINIZ:



Tebrikler! 20. Dersi tamamlayarak telefonunuzun veya güvenlik kameralarının Yüz Tespiti yaparken arka planda nasıl çalıştığını birebir deneyimlediniz.

Bir sonraki derste, sadece 'yüz' veya 'nesne' bulmaktan çıkıp, o nesnenin NE OLDUĞUNU (Kedi mi? Araba mı? İnsan mı?) anlayan devasa Derin Öğrenme (Deep Learning) modellerine geçiş yapacağız.